In [1]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 96.8 MB/s  0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [typer]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [roboflow]2/3 [roboflow]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [3]:
import os
import time
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from roboflow import Roboflow
from datetime import datetime
from ultralytics import YOLO
import json

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [4]:
API_KEY='lg2xMKLHOFyP2kzsMnCw'
rf=Roboflow(api_key=API_KEY)
project=rf.project("road-damage-l1ju7-adtzs")
version=project.version(1)
dataset=version.download("yolov11",location='dataset')
print('completed successfully',dataset.location)


Extracting Dataset Version Zip to dataset in yolov11:: 100%|██████████| 13518/13518 [00:03<00:00, 4376.69it/s] 


completed successfully /kaggle/working/dataset


In [5]:
DATA_YAML='/kaggle/working/dataset/data.yaml'
EXPERIMENTS_DIR='/kaggle/working/experiments'
OUTPUT_DIR='/kaggle/working/outputs'
RESULTS_JSON='/kaggle/working/outputs/results.json'
PLOTS_DIR='/kaggle/working/outputs/plots'
VAL_IMAGES_DIR='/kaggle/working/dataset/valid/images'
VALD_LABELS_DIR='/kaggle/working/dataset/valid/labels'
PREDICTIONS_DIR='/kaggle/working/outputs/predictions'
IOU_THRESHOLD=0.5
NUM_IMAGES_TO_SHOW=9

In [6]:
MODELS=['yolov8n.pt','yolo11s.pt','yolo11n.pt']

In [7]:
EPOCHS=50
BATCH_SIZE=32
IMAGE_SIZE=640
LEARNING_RATE=(0.001)
OPTIMIZER='Adam'
SEED=22


In [8]:

def train_one_model(weights_name: str) -> dict:

    model_name = weights_name.replace(".pt", "")
    run_name = f"{model_name}_improved_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    print(f"\n=== Start Improved Training: {model_name} ===")

    model = YOLO(weights_name)

    model.train(
        # Dataset
        data=DATA_YAML,

        # Training
        epochs=100,
        batch=BATCH_SIZE,
        imgsz=832,
        patience=25,

        # Optimizer
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        cos_lr=True,

        # Augmentation
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.3,

        degrees=5.0,
        translate=0.1,
        scale=0.5,
        shear=2.0,
        perspective=0.0001,

        fliplr=0.5,
        flipud=0.0,

        mosaic=1.0,
        mixup=0.05,
        close_mosaic=10,

        # Performance
        amp=True,
        pretrained=True,

        # Reproducibility
        seed=SEED,
        deterministic=True,

        # Saving
        project=EXPERIMENTS_DIR,
        name=run_name,
        plots=True,
        save=True,
        verbose=True
    )

    # Load best model
    best_weights_path = os.path.join(
        EXPERIMENTS_DIR,
        run_name,
        "weights",
        "best.pt"
    )

    best_model = YOLO(best_weights_path)

    # Validation
    metrics = best_model.val(
        data=DATA_YAML,
        split="val",
        imgsz=832,
        batch=BATCH_SIZE,
        plots=True
    )

    result = {
        "model_name": model_name,
        "best_weights_path": best_weights_path,
        "mAP50": round(metrics.box.map50, 4),
        "mAP50-95": round(metrics.box.map, 4),
        "precision": round(metrics.box.mp, 4),
        "recall": round(metrics.box.mr, 4),
        "model_size_mb": round(
            os.path.getsize(best_weights_path) / (1024 * 1024), 2
        )
    }

    print("\n=== Improved Model Results ===")
    for key, value in result.items():
        print(f"{key}: {value}")

    return result


In [9]:
def train_all_model():
    results=[]
    for weights_name in MODELS:
        result=train_one_model(weights_name)
        results.append(result)
    return results    

In [14]:
def print_comparison_table(results: list):
    print("\n=== Comparison Table ===")
    header = f"{'Model':<10}{'mAP50':<10}{'mAP50-95':<12}{'Precision':<12}{'Recall':<10}{'Size(MB)':<10}"
    print(header)
    print("-" * len(header))
    for r in results:
        print(
            f"{r['model_name']:<10}{r['mAP50']:<10}{r['mAP50-95']:<12}"
            f"{r['precision']:<12}{r['recall']:<10}{r['model_size_mb']:<10}"
        )
    best = max(results, key=lambda r: r["mAP50-95"])
    print(f"\nBest model for deployment (highest mAP50-95): {best['model_name']}")
    print(f"Weights path: {best['best_weights_path']}")

In [11]:
def save_results(results: list):
    os.makedirs(os.path.dirname(RESULTS_JSON), exist_ok=True)
    with open(RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\nsaved result: {RESULTS_JSON}")

In [12]:
all_results=train_all_model()



=== Start Improved Training: yolov8n ===
Ultralytics 8.4.197 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=5.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=832, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_impr

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      6.44G      1.937      2.648      2.046        128        832: 100% ━━━━━━━━━━━━ 154/154 1.6it/s 1:340.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 1.3it/s 13.7s0.5s
                   all       1142       3570      0.254     0.0734      0.054     0.0209

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      6.53G      1.919      2.553      2.022        168        832: 100% ━━━━━━━━━━━━ 154/154 1.7it/s 1:330.5sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 1.3it/s 14.1s0.5s
                   all       1142       3570      0.145      0.167      0.094     0.0354

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      6.52G      1.906      2.544      2.006        161        832: 100% ━━━━━━━━━━━━ 154/154 1.6it/s 1:340.4sss
                 Class     Im

In [15]:
print_comparison_table(all_results)


=== Comparison Table ===
Model     mAP50     mAP50-95    Precision   Recall    Size(MB)  
----------------------------------------------------------------
yolov8n   0.4207    0.2338      0.6031      0.4297    6.25      
yolo11s   0.4325    0.2471      0.6293      0.4443    18.34     
yolo11n   0.4319    0.2464      0.6494      0.4116    5.29      

Best model for deployment (highest mAP50-95): yolo11s
Weights path: /kaggle/working/experiments/yolo11s_improved_20261101_141711/weights/best.pt


In [16]:
save_results(all_results)


saved result: /kaggle/working/outputs/results.json
